# 🧪 Sesión 16 · OAuth2 Password Flow y Header Authorization Bearer

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Por qué el estándar OAuth2 usa un formulario y no JSON para el login?

---

## 🖥️ Así se verá el resultado final

```http
POST /token
Content-Type: application/x-www-form-urlencoded
username=ana&password=secreta
HTTP/1.1 200 OK
{"access_token":"...","token_type":"bearer"}

GET /api/perfil   Authorization: Bearer ...
HTTP/1.1 200 OK
{"usuario":"ana"}

GET /api/perfil   (sin token)
HTTP/1.1 401 Unauthorized
WWW-Authenticate: Bearer
```

_El formulario y el encabezado no son un capricho: son el contrato que esperan los clientes OAuth2._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Implementar el Password Flow de OAuth2 con FastAPI.
- Enviar y validar el encabezado Authorization: Bearer.
- Responder 401 con el reto WWW-Authenticate correcto.

## ✅ Criterios de aceptación

El verificador ejecutará los casos HTTP y comprobará los archivos del contrato. Todo debe quedar en verde:

- [ ] El formulario de login emite token → HTTP 200
- [ ] El token Bearer abre el perfil → HTTP 200
- [ ] Sin token se responde 401 con WWW-Authenticate → HTTP 401
- [ ] Un esquema distinto se rechaza → HTTP 401
- [ ] Se usa el flujo OAuth2 de FastAPI

## 🧩 El reto

Implementa el Password Flow de OAuth2 con formulario y protege una ruta con OAuth2PasswordBearer.

**Requisitos del contrato**

- El usuario de prueba es ana con contraseña secreta.
- POST /token recibe username y password como formulario, no como JSON.
- Responde 200 con access_token y token_type bearer.
- GET /api/perfil exige Authorization: Bearer y responde 200 con el usuario.
- Sin token responde 401 e incluye WWW-Authenticate: Bearer.
- Un esquema distinto de Bearer también responde 401.

> **Pista conceptual:** `OAuth2PasswordRequestForm` se encarga del formulario; `OAuth2PasswordBearer` se encarga del encabezado y del 401 con el reto correcto.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion16.md` con la diferencia entre el formulario y el encabezado.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


In [2]:
import os
import secrets

os.environ["JWT_SECRET_KEY"] = secrets.token_urlsafe(32)

print("Clave secreta configurada correctamente")

Clave secreta configurada correctamente


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [3]:

%%writefile app.py

import os
from datetime import datetime, timedelta, timezone

from fastapi import FastAPI, Depends, HTTPException
from fastapi.security import (
    OAuth2PasswordBearer,
    OAuth2PasswordRequestForm
)
from jose import jwt, JWTError

# Crear la aplicación FastAPI
app = FastAPI(
    title="API OAuth2 Password Flow",
    version="1.0.0"
)

# Obtener la clave secreta desde el entorno
SECRET_KEY = os.getenv("JWT_SECRET_KEY")

if not SECRET_KEY:
    raise RuntimeError("Falta configurar JWT_SECRET_KEY")

# Configuración JWT
ALGORITHM = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30

# Usuario de prueba
USUARIO_PRUEBA = "ana"
PASSWORD_PRUEBA = "secreta"

# Herramienta OAuth2 para recibir el token Bearer
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="token")


# Función para generar el token
def crear_token(usuario: str):

    ahora = datetime.now(timezone.utc)

    expiracion = ahora + timedelta(
        minutes=ACCESS_TOKEN_EXPIRE_MINUTES
    )

    datos_token = {
        "usuario": usuario,
        "exp": expiracion
    }

    token = jwt.encode(
        datos_token,
        SECRET_KEY,
        algorithm=ALGORITHM
    )

    return token


# Endpoint para iniciar sesión mediante formulario
@app.post("/token")
def login(
    formulario: OAuth2PasswordRequestForm = Depends()
):

    if (
        formulario.username != USUARIO_PRUEBA
        or formulario.password != PASSWORD_PRUEBA
    ):
        raise HTTPException(
            status_code=401,
            detail="Credenciales incorrectas",
            headers={"WWW-Authenticate": "Bearer"}
        )

    access_token = crear_token(formulario.username)

    return {
        "access_token": access_token,
        "token_type": "bearer"
    }


# Endpoint protegido para consultar el perfil
@app.get("/api/perfil")
def obtener_perfil(
    token: str = Depends(oauth2_scheme)
):

    try:
        payload = jwt.decode(
            token,
            SECRET_KEY,
            algorithms=[ALGORITHM]
        )

        usuario = payload.get("usuario")

        if not usuario or not isinstance(usuario, str):
            raise HTTPException(
                status_code=401,
                detail="Token inválido",
                headers={"WWW-Authenticate": "Bearer"}
            )

    except JWTError:
        raise HTTPException(
            status_code=401,
            detail="Token inválido o expirado",
            headers={"WWW-Authenticate": "Bearer"}
        )

    return {"usuario": usuario}



Writing app.py


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Python 3.10 o superior, FastAPI, Pydantic y Google Colab. Se utilizará python-multipart para procesar formularios y Python-JOSE para los tokens JWT. |
| **Rol** | Arquitecto de software especializado en APIs RESTful, OAuth2 y seguridad web. |
| **Especificación** | Implementar POST /token con OAuth2PasswordRequestForm para recibir username y password como formulario. Validar las credenciales de ana y secreta. Generar un JWT con expiración. Proteger GET /api/perfil con OAuth2PasswordBearer. Sin token o con un esquema distinto de Bearer, responder HTTP 401 y utilizar el encabezado WWW-Authenticate: Bearer. |
| **Output** | Código completo y modular en app.py, con herramientas OAuth2 de FastAPI, generación y validación JWT y respuestas HTTP correctas. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [5]:
from verificar import probar_http

casos = [{'descripcion': 'El formulario de login emite token',
  'metodo': 'POST',
  'ruta': '/token',
  'data': {'username': 'ana', 'password': 'secreta'},
  'estado': 200,
  'json_contiene': {'token_type': 'bearer'},
  'guardar': {'token': 'access_token'}},
 {'descripcion': 'El token Bearer abre el perfil',
  'metodo': 'GET',
  'ruta': '/api/perfil',
  'headers': {'Authorization': 'Bearer {token}'},
  'estado': 200,
  'json_contiene': {'usuario': 'ana'}},
 {'descripcion': 'Sin token se responde 401 con WWW-Authenticate',
  'metodo': 'GET',
  'ruta': '/api/perfil',
  'estado': 401,
  'headers_contiene': {'www-authenticate': 'bearer'}},
 {'descripcion': 'Un esquema distinto se rechaza',
  'metodo': 'GET',
  'ruta': '/api/perfil',
  'headers': {'Authorization': 'Basic YW5hOnNlY3JldGE='},
  'estado': 401}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


VERIFICACIÓN DEL RESULTADO · 4 casos · app.py
✅ 01. El formulario de login emite token
✅ 02. El token Bearer abre el perfil
✅ 03. Sin token se responde 401 con WWW-Authenticate
✅ 04. Un esquema distinto se rechaza
------------------------------------------------------------------------------
RESULTADO: 4/4 casos correctos. ¡Resultado logrado!


In [9]:
from verificar import probar_http

casos = [
    {
        "descripcion": "El formulario de login emite token",
        "metodo": "POST",
        "ruta": "/token",
        "data": {
            "username": "ana",
            "password": "secreta"
        },
        "estado": 200,
        "json_contiene": {
            "token_type": "bearer"
        },
        "guardar": {
            "token": "access_token"
        }
    },
    {
        "descripcion": "El token Bearer abre el perfil",
        "metodo": "GET",
        "ruta": "/api/perfil",
        "headers": {
            "Authorization": "Bearer {token}"
        },
        "estado": 200,
        "json_contiene": {
            "usuario": "ana"
        }
    },
    {
        "descripcion": "Sin token se rechaza el acceso",
        "metodo": "GET",
        "ruta": "/api/perfil",
        "estado": 401,
        "headers_contiene": {
            "www-authenticate": "bearer"
        }
    },
    {
        "descripcion": "Un esquema distinto de Bearer se rechaza",
        "metodo": "GET",
        "ruta": "/api/perfil",
        "headers": {
            "Authorization": "Basic abc123"
        },
        "estado": 401,
        "headers_contiene": {
            "www-authenticate": "bearer"
        }
    }
]

probar_http(
    casos,
    archivo="app.py",
    limpiar=[],
    env={}
)


VERIFICACIÓN DEL RESULTADO · 4 casos · app.py
✅ 01. El formulario de login emite token
✅ 02. El token Bearer abre el perfil
✅ 03. Sin token se rechaza el acceso
✅ 04. Un esquema distinto de Bearer se rechaza
------------------------------------------------------------------------------
RESULTADO: 4/4 casos correctos. ¡Resultado logrado!


In [7]:
from verificar import revisar_archivo, resumen

resultados = []

resultados.append(
    revisar_archivo(
        "app.py",
        regex=[
            "OAuth2PasswordRequestForm",
            "OAuth2PasswordBearer"
        ],
        descripcion="Se utiliza OAuth2 de FastAPI"
    )
)

resumen(resultados)

✅ Se utiliza OAuth2 de FastAPI
------------------------------------------------------------------------------
RESULTADO: 1/1 comprobaciones correctas. ¡Resultado logrado!


In [6]:
from verificar import revisar_archivo, resumen

resultados = []
resultados.append(revisar_archivo("app.py", regex=["OAuth2PasswordBearer", "OAuth2PasswordRequestForm"], descripcion="Se usa el flujo OAuth2 de FastAPI"))
resumen(resultados)


✅ Se usa el flujo OAuth2 de FastAPI
------------------------------------------------------------------------------
RESULTADO: 1/1 comprobaciones correctas. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion16.md` a `auditorias/auditoria_sesion16.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué? R.- Lo que más me costó fue comprender la diferencia entre enviar las credenciales mediante un formulario y enviar el token mediante el encabezado Authorization. Al principio pensaba que ambos se enviaban de la misma manera, pero aprendí que cada uno cumple una función diferente.
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato? R.- Decidí separar la creación del token en una función llamada crear_token() y establecer una duración de 30 minutos. También utilicé una variable de entorno para guardar la clave secreta y mantener el código organizado.
   - ¿Qué error detectó la verificación que tú no habías previsto? R.- Una situación que tuve que considerar fue que no bastaba con rechazar la petición cuando faltaba el token. También era necesario enviar el encabezado WWW-Authenticate: Bearer para cumplir correctamente con el estándar OAuth2.
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
